In [1]:
!pip install gradio

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import gradio as gr

print(gr.__version__)

6.32.0


In [6]:
import gradio as gr
import random
import os
from datetime import datetime

FILE_NAME = "tasks.txt"


# ---------------- FILE CREATION ---------------- #

def create_file():
    if not os.path.exists(FILE_NAME):
        open(FILE_NAME, "w").close()


# ---------------- LOAD DATA ---------------- #

def get_tasks():
    create_file()

    tasks = []

    with open(FILE_NAME, "r") as file:
        for line in file:
            data = line.strip().split("|")

            if len(data) == 4:
                tasks.append(data)

    return tasks


# ---------------- ADD TASK ---------------- #

def add_task(name, date, task):

    try:

        if not name or not date or not task:
            return "❌ Please fill all fields", get_tasks()

        with open(FILE_NAME, "a") as file:
            file.write(f"{name}|{date}|{task}|Pending\n")

        return "✅ Task Added Successfully", get_tasks()

    except Exception as e:
        return str(e), get_tasks()


# ---------------- COMPLETE TASK ---------------- #

def complete_task(index):

    try:

        index = int(index)

        with open(FILE_NAME, "r") as file:
            lines = file.readlines()

        if index < 0 or index >= len(lines):
            return "❌ Invalid Task ID", get_tasks()

        data = lines[index].strip().split("|")

        data[3] = "Completed"

        lines[index] = "|".join(data) + "\n"

        with open(FILE_NAME, "w") as file:
            file.writelines(lines)

        return "✅ Task Completed", get_tasks()

    except Exception as e:
        return str(e), get_tasks()


# ---------------- DELETE TASK ---------------- #

def delete_task(index):

    try:

        index = int(index)

        with open(FILE_NAME, "r") as file:
            lines = file.readlines()

        if index < 0 or index >= len(lines):
            return "❌ Invalid Task ID", get_tasks()

        del lines[index]

        with open(FILE_NAME, "w") as file:
            file.writelines(lines)

        return "🗑️ Task Deleted", get_tasks()

    except Exception as e:
        return str(e), get_tasks()


# ---------------- SEARCH USER ---------------- #

def search_user(name):

    try:

        result = []

        with open(FILE_NAME, "r") as file:

            for line in file:

                data = line.strip().split("|")

                if data[0].lower() == name.lower():
                    result.append(data)

        return result

    except:
        return []


# ---------------- ANALYZE PRODUCTIVITY ---------------- #

def analyze_productivity():

    try:

        total = 0
        completed = 0

        with open(FILE_NAME, "r") as file:

            for line in file:

                total += 1

                data = line.strip().split("|")

                if data[3] == "Completed":
                    completed += 1

        if total == 0:
            return "No Tasks Available"

        percentage = (completed / total) * 100

        if percentage >= 90:
            badge = "🏆 GOLD ACHIEVER"
            message = "Excellent! Outstanding productivity."

        elif percentage >= 75:
            badge = "🥈 SILVER PERFORMER"
            message = "Great Work! Keep Going!"

        elif percentage >= 50:
            badge = "🥉 BRONZE ACHIEVER"
            message = "Good Effort! Keep Improving!"

        else:

            quotes = [
                "Never give up. Great things take time.",
                "Success starts with small efforts.",
                "Every task completed is progress.",
                "Believe in yourself.",
                "Tomorrow is another opportunity."
            ]

            badge = "💪 KEEP TRYING"
            message = random.choice(quotes)

        report = f"""
SMART PRODUCTIVITY REPORT

Total Tasks : {total}

Completed Tasks : {completed}

Productivity Score : {percentage:.2f}%

Achievement : {badge}

Message :
{message}
"""

        return report

    except Exception as e:
        return str(e)


# ---------------- GRADIO UI ---------------- #

with gr.Blocks(
    theme=gr.themes.Soft(),
    title="Smart To-Do Task Manager"
) as demo:

    gr.Markdown(
        """
        # 🚀 Smart To-Do Task Manager
        
        ### Productivity Analysis & Motivation System
        """
    )

    with gr.Row():

        name = gr.Textbox(
            label="User Name"
        )

        date = gr.Textbox(
            value=datetime.now().strftime("%d-%m-%Y"),
            label="Date"
        )

    task = gr.Textbox(
        label="Task"
    )

    with gr.Row():

        add_btn = gr.Button(
            "➕ Add Task",
            variant="primary"
        )

        analyze_btn = gr.Button(
            "📊 Analyze Productivity"
        )

    status = gr.Textbox(
        label="Status"
    )

    task_table = gr.Dataframe(
        headers=["Name", "Date", "Task", "Status"],
        value=get_tasks(),
        interactive=False,
        label="Task Records"
    )

    gr.Markdown("## ✅ Complete Task")

    complete_id = gr.Number(
        label="Task ID (Row Number Starting from 0)"
    )

    complete_btn = gr.Button("Mark Complete")

    gr.Markdown("## 🗑 Delete Task")

    delete_id = gr.Number(
        label="Task ID"
    )

    delete_btn = gr.Button("Delete Task")

    gr.Markdown("## 🔍 Search User")

    search_name = gr.Textbox(
        label="Enter User Name"
    )

    search_btn = gr.Button("Search")

    search_result = gr.Dataframe(
        headers=["Name", "Date", "Task", "Status"],
        interactive=False
    )

    report = gr.Textbox(
        label="Performance Report",
        lines=12
    )

    # Events

    add_btn.click(
        add_task,
        inputs=[name, date, task],
        outputs=[status, task_table]
    )

    complete_btn.click(
        complete_task,
        inputs=complete_id,
        outputs=[status, task_table]
    )

    delete_btn.click(
        delete_task,
        inputs=delete_id,
        outputs=[status, task_table]
    )

    search_btn.click(
        search_user,
        inputs=search_name,
        outputs=search_result
    )

    analyze_btn.click(
        analyze_productivity,
        outputs=report
    )

demo.launch()

C:\Users\rakes\AppData\Local\Temp\ipykernel_10716\1315334500.py:201: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.
